# 03 — SVM Baseline

This notebook trains and evaluates the SVM baseline using `src/baseline.py`.

The baseline establishes the **performance floor the CNN must exceed**.  
Expected accuracy: **60–75 %** (macro F1 in the same range).

**Contents:**
1. Load the spectrogram dataset and split it into train / val / test.
2. Flatten spectrograms into 22 144-feature vectors.
3. Fit an RBF-kernel SVM pipeline.
4. Evaluate on the held-out test set — macro F1, accuracy, per-class report, and confusion matrix.

> **Pre-requisite:** `data/spectrograms/labels.csv` must exist.
> Run `python src/features.py --data-dir data/` first.

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np

from src.baseline import evaluate, load_features, train_svm
from src.config import CLASS_LABELS, NUM_CLASSES
from src.dataset import load_labels, split_dataset

DATA_DIR    = ROOT / "data"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print(f"ROOT        : {ROOT}")
print(f"DATA_DIR    : {DATA_DIR}   (exists={DATA_DIR.exists()})")
print(f"RESULTS_DIR : {RESULTS_DIR}")

In [ ]:
labels_csv = DATA_DIR / "spectrograms" / "labels.csv"

if not labels_csv.exists():
    print(f"WARNING: {labels_csv} not found.")
    print("Run:  python src/features.py --data-dir data/")
    train_df = val_df = test_df = None
else:
    df = load_labels(DATA_DIR / "spectrograms")
    train_df, val_df, test_df = split_dataset(df)
    aug_count = train_df["is_augmented"].sum()
    print(f"Total samples  : {len(df)}")
    print(f"  train        : {len(train_df)}  (of which augmented: {aug_count})")
    print(f"  val          : {len(val_df)}")
    print(f"  test         : {len(test_df)}")
    print(f"\nClass distribution in test split:")
    print(test_df["class_label"].value_counts().reindex(CLASS_LABELS.values()).to_string())

## Section 1 — Feature Loading

Each log-Mel spectrogram has shape **(128, 173)**.  
`load_features(df)` flattens it to a 1-D float32 vector of **128 × 173 = 22 144** features.

The SVM receives no 2-D structure — every frequency bin and time frame is treated as an
independent feature. This is the key limitation that the CNN overcomes with 2-D convolutions.

> The SVM is trained on **non-augmented training samples only**.
> Augmented copies are excluded so the baseline is comparable to the CNN's no-augmentation run.

In [ ]:
if train_df is None:
    print("No data — skipping feature loading.")
    X_train = X_test = y_train = y_test = None
else:
    # Exclude augmented rows — the SVM baseline uses raw spectrograms only.
    train_orig = train_df[~train_df["is_augmented"]].reset_index(drop=True)

    print("Loading training features…")
    X_train, y_train = load_features(train_orig)
    print(f"  X_train : {X_train.shape}   y_train : {y_train.shape}")

    print("Loading test features…")
    X_test, y_test = load_features(test_df)
    print(f"  X_test  : {X_test.shape}    y_test  : {y_test.shape}")

## Section 2 — SVM Training

`train_svm(X_train, y_train)` fits a **`StandardScaler → SVC(kernel='rbf', C=10)`** pipeline.

| Hyperparameter | Value | Rationale |
|---|---|---|
| Kernel | RBF | Handles non-linear boundaries without manual feature engineering |
| `C` | 10 | Moderate regularisation — allows some margin violations |
| `gamma` | `'scale'` | Auto: γ = 1 / (n\_features × X.var()) |
| Scaler | `StandardScaler` | Essential before RBF — features must be on the same scale |

Training on 22 144 features takes a few minutes on CPU for ~1 000 samples.
The pipeline is returned so `evaluate` can call `.predict(X_test)` on raw features
and the scaler applies automatically.

In [ ]:
if X_train is None:
    print("No features loaded — skipping training.")
    model = None
else:
    print(
        f"Training SVM on {X_train.shape[0]} samples "
        f"× {X_train.shape[1]} features…"
    )
    model = train_svm(X_train, y_train)
    print("Done.")

## Section 3 — Evaluation

`evaluate(model, X_test, y_test, output_dir)` returns a dict with three keys:

| Key | Content |
|---|---|
| `macro_f1` | Unweighted mean F1 across all five classes — the **primary metric** |
| `accuracy` | Overall fraction of correctly classified test clips |
| `report` | Full `sklearn` classification report with per-class precision, recall, F1 |

It also writes `baseline_confusion.png` (seaborn heatmap with class-name axes) and
`baseline_report.txt` to the results directory.
The confusion matrix is then loaded and displayed inline below.

In [ ]:
if model is None:
    print("No model — skipping evaluation.")
    results = None
else:
    results = evaluate(model, X_test, y_test, RESULTS_DIR)
    # Macro F1 is the primary metric and is printed first by evaluate().
    print(f"\nSaved to {RESULTS_DIR}:")
    print(f"  baseline_confusion.png")
    print(f"  baseline_report.txt")

In [ ]:
confusion_png = RESULTS_DIR / "baseline_confusion.png"

if results is None:
    print("No results to display.")
elif not confusion_png.exists():
    print(f"Confusion matrix PNG not found at {confusion_png}")
else:
    img = mpimg.imread(str(confusion_png))
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(img)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

## Observations — Baseline as the CNN Benchmark

| Metric | Expected range | What it means |
|---|---|---|
| **Macro F1** | 60–75 % | Primary metric — equal weight per class regardless of sample count |
| **Accuracy** | 60–75 % | Overall fraction correct; can be misleading if classes are imbalanced |

### What the SVM cannot exploit
- **2-D spatial patterns** — frequency bands, temporal structure, and their co-occurrence.
- **Hierarchical features** — combinations of low-level and mid-level patterns.
- **Positional invariance** — the same squeal at t=1 s and t=3 s look different to the SVM.

### What the CNN should exploit
- Shared 3×3 convolutional filters detect local frequency–time patterns at any position.
- Three conv blocks build increasingly abstract representations.
- GlobalAveragePooling2D aggregates spatial information without positional bias.
- BatchNorm + Dropout regularise on a small dataset.

### Failure modes to investigate if CNN ≤ SVM
1. **Data leakage** — augmented samples in val/test (check `dataset.py`).
2. **Overfitting** — training accuracy >> val accuracy; try more Dropout or fewer epochs.
3. **Normalisation** — verify spectrograms are scaled to [-1, 1] before the CNN sees them.